In [2]:
#!/usr/bin/env python
"""Train an XGBoost model to predict French box‑office admissions.

Pipeline steps
--------------
1. Chargement des données depuis `allocine_cleaned_202504151041.csv`
2. Nettoyage :
   * Suppression des colonnes inutiles
   * Suppression des films sans `box_office_fr`
   * Suppression des films sans acteurs
   * Imputation de `duration` par la médiane
   * Imputation de `director`, `writer`, `distributor` par 'unknown'
3. Feature engineering :
   * Encodage basique des champs textuels (`genres`, `title`) via TF‑IDF (facultatif)
   * Création de :
        - `actors_avg_top3` : moyenne des 3 meilleurs historiques d’entrées parmi les acteurs
        - `actors_count_top3` : nombre moyen de films tournés par ces mêmes acteurs
        - `director_avg`, `director_count`
        - `dist_avg`, `dist_count`
4. Séparation train / test (stratified K‑Fold CV  (5))
5. Optimisation Optuna + XGBoost (tree_method='gpu_hist') : RMSE
6. Sauvegarde du modèle pickle + Optuna study
"""

import pandas as pd
import numpy as np
import re
from pathlib import Path
from datetime import datetime
from sklearn.model_selection import KFold, cross_val_score
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from xgboost import XGBRegressor, callback
import optuna
import joblib

DATA_PATH = Path('allocine_cleaned_202504151041.csv')
TARGET = 'box_office_fr'

DROP_COLS = ['original_title', 'press_rating', 'audience_rating',
             'image_url', 'box_office_us', 'synopsis', 'id']

SEED = 42


def load_and_clean(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path, parse_dates=['release_date'])
    # Remove useless cols
    df = df.drop(columns=[c for c in DROP_COLS if c in df.columns])

    # Drop rows without target
    df = df.dropna(subset=[TARGET])

    # Remove rows without actors
    df = df[~df['actors'].isna() & (df['actors'].str.strip() != '')]

    # Impute duration
    duration_median = df['duration'].median()
    df['duration'] = df['duration'].fillna(duration_median)

    # Impute categorical with 'unknown'
    for col in ['director', 'writer', 'distributor']:
        df[col] = df[col].fillna('unknown')

    return df


def parse_list_column(col: pd.Series) -> pd.Series:
    # Transform strings like {"Actor1","Actor2"} -> list
    return col.str.replace('[{}"]', '', regex=True).str.split(',')


def compute_entity_stats(df: pd.DataFrame, entity_col: str) -> pd.DataFrame:
    """Return cumulative mean & count of target for an entity before the current row."""
    stats_avg, stats_cnt = [], []
    # Sort by release_date to avoid leakage; fallback on index
    df_sorted = df.sort_values('release_date').reset_index(drop=True)
    entity_history = {}
    for _, row in df_sorted.iterrows():
        entity = row[entity_col]
        # Ensure iterable
        entities = entity if isinstance(entity, list) else [entity]
        avgs, cnts = [], []
        for e in entities:
            hist = entity_history.get(e, [])
            if hist:
                avgs.append(np.mean(hist))
                cnts.append(len(hist))
        # Compute features
        if avgs:
            stats_avg.append(np.mean(sorted(avgs, reverse=True)[:3]))  # top‑3
            stats_cnt.append(np.mean(sorted(cnts, reverse=True)[:3]))
        else:
            stats_avg.append(np.nan)
            stats_cnt.append(np.nan)
        # Update history
        for e in entities:
            entity_history.setdefault(e, []).append(row[TARGET])
    return pd.Series(stats_avg), pd.Series(stats_cnt)


def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    # Parse list columns
    df['actors_list'] = parse_list_column(df['actors'])

    # Actors features
    actors_avg, actors_cnt = compute_entity_stats(df, 'actors_list')
    df['actors_avg_top3'] = actors_avg
    df['actors_count_top3'] = actors_cnt

    # Director stats (single value)
    dir_avg, dir_cnt = compute_entity_stats(df, 'director')
    df['director_avg'] = dir_avg
    df['director_count'] = dir_cnt

    # Distributor stats
    dist_avg, dist_cnt = compute_entity_stats(df, 'distributor')
    df['dist_avg'] = dist_avg
    df['dist_count'] = dist_cnt

    # Replace possible NaN resulting from no history with global median/0
    df['actors_avg_top3'] = df['actors_avg_top3'].fillna(df[TARGET].median())
    df['actors_count_top3'] = df['actors_count_top3'].fillna(0)
    df['director_avg'] = df['director_avg'].fillna(df[TARGET].median())
    df['director_count'] = df['director_count'].fillna(0)
    df['dist_avg'] = df['dist_avg'].fillna(df[TARGET].median())
    df['dist_count'] = df['dist_count'].fillna(0)

    return df


def build_preprocessor(df: pd.DataFrame):
    categorical_cols = ['genres', 'director', 'writer', 'distributor']
    numeric_cols = ['duration', 'actors_avg_top3', 'actors_count_top3',
                    'director_avg', 'director_count', 'dist_avg', 'dist_count']

    # One‑hot encode categorical, keeping top 50 frequent to limit sparsity
    encoder = OneHotEncoder(handle_unknown='ignore', min_frequency=20)

    preprocessor = ColumnTransformer(
        transformers=[
            ('cat', encoder, categorical_cols),
            ('num', 'passthrough', numeric_cols)
        ])
    return preprocessor


def objective(trial: optuna.Trial, X, y, preprocessor):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 300, 2000),
        'max_depth': trial.suggest_int('max_depth', 4, 12),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'reg_alpha': trial.suggest_float('reg_alpha', 0, 5.0),
        'reg_lambda': trial.suggest_float('reg_lambda', 0, 5.0),
        'tree_method': 'gpu_hist',
        'predictor': 'gpu_predictor',
        'random_state': SEED,
    }
    model = XGBRegressor(**params)
    pipe = Pipeline(steps=[('prep', preprocessor), ('model', model)])
    cv = KFold(n_splits=5, shuffle=True, random_state=SEED)
    scores = -cross_val_score(pipe, X, y, cv=cv,
                              scoring='neg_root_mean_squared_error',
                              n_jobs=-1)
    return scores.mean()


def optimise(X, y, preprocessor, n_trials=50):
    study = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=SEED))
    func = lambda trial: objective(trial, X, y, preprocessor)
    study.optimize(func, n_trials=n_trials, timeout=3600, show_progress_bar=True)
    return study


def train_best_model(X, y, preprocessor, study):
    best_params = study.best_trial.params
    best_params.update({'tree_method': 'gpu_hist',
                        'predictor': 'gpu_predictor',
                        'random_state': SEED})
    model = XGBRegressor(**best_params)
    pipe = Pipeline(steps=[('prep', preprocessor), ('model', model)])
    pipe.fit(X, y)
    return pipe


def main():
    print("\n=== Chargement et préparation des données ===")
    df = load_and_clean(DATA_PATH)
    df = engineer_features(df)

    y = df[TARGET]
    X = df.drop(columns=[TARGET, 'actors', 'actors_list'])  # keep cleaned

    preprocessor = build_preprocessor(df)

    print("\n=== Optimisation Optuna ===")
    study = optimise(X, y, preprocessor, n_trials=100)
    print("Meilleur RMSE moyen CV :", study.best_value)
    print("Paramètres :", study.best_trial.params)

    print("\n=== Entraînement modèle final ===")
    final_model = train_best_model(X, y, preprocessor, study)

    out_path = Path('xgb_boxoffice_fr.pkl')
    joblib.dump(final_model, out_path)
    print(f"Modèle sauvegardé à {out_path.resolve()}")

    study_path = Path('optuna_study.pkl')
    joblib.dump(study, study_path)
    print(f"Study sauvegardée à {study_path.resolve()}")


if __name__ == '__main__':
    main()



=== Chargement et préparation des données ===


[I 2025-04-24 01:20:10,269] A new study created in memory with name: no-name-ff364933-e99f-42ee-a1a8-f86077823aa8



=== Optimisation Optuna ===


  0%|          | 0/100 [00:00<?, ?it/s]

/home/maxime/DataDevIA/predictioncinema/popularity_movie_prediction_project/ML/.venv/lib/python3.12/site-packages/xgboost/core.py:160: UserWarning: [01:20:12] WARNING: /workspace/src/common/error_msg.cc:27: The tree method `gpu_hist` is deprecated since 2.0.0. To use GPU training, set the `device` parameter to CUDA instead.

    E.g. tree_method = "hist", device = "cuda"

  warnings.warn(smsg, UserWarning)
/home/maxime/DataDevIA/predictioncinema/popularity_movie_prediction_project/ML/.venv/lib/python3.12/site-packages/xgboost/core.py:160: UserWarning: [01:20:12] WARNING: /workspace/src/common/error_msg.cc:27: The tree method `gpu_hist` is deprecated since 2.0.0. To use GPU training, set the `device` parameter to CUDA instead.

    E.g. tree_method = "hist", device = "cuda"

  warnings.warn(smsg, UserWarning)
/home/maxime/DataDevIA/predictioncinema/popularity_movie_prediction_project/ML/.venv/lib/python3.12/site-packages/xgboost/core.py:160: UserWarning: [01:20:12] WARNING: /workspace/s

[W 2025-04-24 01:20:12,429] Trial 0 failed with parameters: {'n_estimators': 937, 'max_depth': 12, 'learning_rate': 0.1205712628744377, 'subsample': 0.7993292420985183, 'colsample_bytree': 0.5780093202212182, 'min_child_weight': 2, 'reg_alpha': 0.2904180608409973, 'reg_lambda': 4.330880728874676} because of the following error: ValueError('\nAll the 5 fits failed.\nIt is very likely that your model is misconfigured.\nYou can try to debug the error by setting error_score=\'raise\'.\n\nBelow are more details about the failures:\n--------------------------------------------------------------------------------\n1 fits failed with the following error:\nTraceback (most recent call last):\n  File "/home/maxime/DataDevIA/predictioncinema/popularity_movie_prediction_project/ML/.venv/lib/python3.12/site-packages/sklearn/model_selection/_validation.py", line 866, in _fit_and_score\n    estimator.fit(X_train, y_train, **fit_params)\n  File "/home/maxime/DataDevIA/predictioncinema/popularity_movie_

ValueError: 
All the 5 fits failed.
It is very likely that your model is misconfigured.
You can try to debug the error by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
1 fits failed with the following error:
Traceback (most recent call last):
  File "/home/maxime/DataDevIA/predictioncinema/popularity_movie_prediction_project/ML/.venv/lib/python3.12/site-packages/sklearn/model_selection/_validation.py", line 866, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "/home/maxime/DataDevIA/predictioncinema/popularity_movie_prediction_project/ML/.venv/lib/python3.12/site-packages/sklearn/base.py", line 1389, in wrapper
    return fit_method(estimator, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/maxime/DataDevIA/predictioncinema/popularity_movie_prediction_project/ML/.venv/lib/python3.12/site-packages/sklearn/pipeline.py", li